# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asmajavaid1270/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


**Unit of Analysis:** One row represents a single (query, url, date) or (query, date) grain depending on your dataset level.

**Time Window:** Observed data spans from YYYY-MM-DD to YYYY-MM-DD (e.g., last 90 days of Search Console / SERP data).

In [12]:
import pandas as pd

# Assuming your DataFrame is 'df'
# Placeholder for df - you should replace this with your actual data loading code
df = pd.DataFrame({
    'date': pd.to_datetime(['2023-01-01', '2023-01-01', '2023-01-02', '2023-01-02']),
    'query': ['query_a', 'query_b', 'query_a', 'query_c'],
    'url': ['url_1', 'url_2', 'url_1', 'url_3'],
    'impressions': [100, 150, 120, 90],
    'clicks': [10, 15, 12, 9],
    'ctr': [0.1, 0.1, 0.1, 0.1],
    'position': [1, 2, 1, 3],
    'internal_id': [101, 102, 103, 104]
})

print("Min Date:", df['date'].min())
print("Max Date:", df['date'].max())
print("Total Rows:", len(df))
print("Unique Grain Rows:", df.groupby(['query', 'date', 'url']).ngroups)

Min Date: 2023-01-01 00:00:00
Max Date: 2023-01-02 00:00:00
Total Rows: 4
Unique Grain Rows: 4


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Features:** impressions, clicks, ctr, historical_avg_position

**Label / Target:** position (or rank_change, click_through_class)

**Context:** query, url, device, country, date

**Excluded:** internal_id, raw_html — Reason: High cardinality / redundant metadata with no predictive signal.

In [13]:
# Verify column presence and data types
features = ['impressions', 'clicks', 'ctr']
label = ['position']
context = ['query', 'url', 'date']
excluded = ['internal_id']

all_fields = features + label + context + excluded
print("All fields accounted for:", set(all_fields).issubset(df.columns))

All fields accounted for: True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:
# Check 1: Grain Duplicate Check
# Redefine fields locally for robustness against execution order
features = ['impressions', 'clicks', 'ctr']
label = ['position']
context = ['query', 'url', 'date']
excluded = ['internal_id']

all_fields = features + label + context + excluded

duplicates = df.duplicated(subset=['query', 'url', 'date']).sum()
print(f"Duplicate Grain Count: {duplicates}")

# Check 2: Missing Values Check
print("\nMissing Values Count:")
print(df[all_fields].isnull().sum())

# Check 3: Row Counts & Summary Statistics
print("\nDataset Summary:")
print(df.describe())

Duplicate Grain Count: 0

Missing Values Count:
impressions    0
clicks         0
ctr            0
position       0
query          0
url            0
date           0
internal_id    0
dtype: int64

Dataset Summary:
                      date  impressions     clicks  ctr  position  internal_id
count                    4     4.000000   4.000000  4.0  4.000000     4.000000
mean   2023-01-01 12:00:00   115.000000  11.500000  0.1  1.750000   102.500000
min    2023-01-01 00:00:00    90.000000   9.000000  0.1  1.000000   101.000000
25%    2023-01-01 00:00:00    97.500000   9.750000  0.1  1.000000   101.750000
50%    2023-01-01 12:00:00   110.000000  11.000000  0.1  1.500000   102.500000
75%    2023-01-02 00:00:00   127.500000  12.750000  0.1  2.250000   103.250000
max    2023-01-02 00:00:00   150.000000  15.000000  0.1  3.000000   104.000000
std                    NaN    26.457513   2.645751  0.0  0.957427     1.290994


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**GSC Sampling & Anonymization:** Very low-volume or long-tail queries are hidden by Google for privacy.

**No User Intent/Behavior Post-Click:** Data measures search impressions and clicks only, not dwell time or conversion.

**Window Boundary Effects:** Rolling feature averages near the start date YYYY-MM-DD may suffer from truncated history.

In [15]:
# Verify low-impression/missing tail limit or missing position distribution
print("Percentage of queries with 1 click:", (df['clicks'] == 1).mean() * 100)

Percentage of queries with 1 click: 0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.